In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [ ]:
from team_wtm.cleaning import load_data, merge_data

train_values, train_labels = load_data()
train_data = merge_data(
    train_values,
    train_labels,
    how="outer",
    keep_merge_indicator=True,
)

In [ ]:
merge_counts = train_data["_merge"].value_counts()
display(merge_counts.rename("Rows by merge outcome"))
unmatched_count = int((train_data["_merge"] != "both").sum())
print("Unmatched IDs:", unmatched_count)
if unmatched_count:
    raise ValueError("Resolve unmatched training IDs before modelling.")
train_data = train_data.drop(columns="_merge")

In [ ]:
print("Merged shape:", train_data.shape)
print("Unique IDs:", train_data["id"].nunique())
print("\nTarget counts:")
display(train_data["status_group"].value_counts(dropna=False))

#  Identifying identical feature records and conflicting label

In [ ]:
feature_cols = [
    col for col in train_data.columns
    if col not in ["id", "status_group"]
]

print("Number of feature columns:", len(feature_cols))

In [ ]:
#Check for duplicate rows, excluding id

# Select all columns except the ID
columns_without_id = [
    col for col in train_data.columns
    if col != "id"
]

# Identify all rows that have an identical match, excluding ID
duplicate_rows = train_data[
    train_data.duplicated(
        subset=columns_without_id,
        keep=False
    )
].sort_values(by=columns_without_id)

print("Number of rows involved in duplicates:", len(duplicate_rows))

display(duplicate_rows)

In [ ]:
#Identify identical feature records and conflicting labels


# 1. Select all feature columns, excluding ID and target
feature_cols = [
    col for col in train_data.columns
    if col not in ["id", "status_group"]
]

# 2. Group records with identical feature values
grouped = train_data.groupby(
    feature_cols,
    dropna=False,
    sort=False
)

# 3. Summarise each feature group
group_summary = grouped.agg(
    number_of_records=("id", "size"),
    number_of_unique_labels=("status_group", "nunique")
).reset_index()

# 4. Keep only repeated feature groups
repeated_groups = group_summary[
    group_summary["number_of_records"] > 1
]

# 5. Identify groups with conflicting target labels
conflicting_groups = repeated_groups[
    repeated_groups["number_of_unique_labels"] > 1
]

# 6. Display the summary
print("Number of repeated feature groups:", len(repeated_groups))
print(
    "Total observations in repeated groups:",
    repeated_groups["number_of_records"].sum()
)
print("Groups with conflicting labels:", len(conflicting_groups))

# 7. Display repeated groups and conflicting groups
display(repeated_groups)
display(conflicting_groups)

# 8. Display the original records with conflicting labels
conflicting_records = train_data.merge(
    conflicting_groups[feature_cols],
    on=feature_cols,
    how="inner"
)

display(
    conflicting_records[
        ["id", "status_group"] + feature_cols
    ]
)

In [ ]:
print("Records in conflicting group:", len(conflicting_records))
print(
    conflicting_records["status_group"]
    .value_counts()
)

In [ ]:
# 1. Count duplicates across all columns
duplicates_all_columns = train_data.duplicated().sum()

# 2. Select feature columns, excluding ID and target
feature_cols = [
    col for col in train_data.columns
    if col not in ["id", "status_group"]
]

# 3. Count duplicates based on feature columns only
duplicates_features = train_data.duplicated(
    subset=feature_cols
).sum()

# 4. Display the results
print("Duplicates across all columns:", duplicates_all_columns)
print("Duplicates based on features only:", duplicates_features)

#Now identify groups containing conflicting labels.

#Applying the cleaning rules

In [ ]:
from team_wtm.cleaning import clean_data as apply_cleaning_rules, derive_pump_age
from team_wtm.features import add_feature_columns

clean_data = apply_cleaning_rules(train_data)
clean_data["pump_age"] = derive_pump_age(clean_data)
clean_data = add_feature_columns(clean_data)

print("Cleaned dimensions:", clean_data.shape)
print("Missing pump ages:", int(clean_data["pump_age"].isna().sum()))

In [ ]:
#checking missing values and data types
missing_summary = pd.DataFrame({
    "missing_count": clean_data.isna().sum(),
    "missing_percent": clean_data.isna().mean().mul(100).round(2),
    "data_type": clean_data.dtypes.astype(str)
})

missing_summary = missing_summary.sort_values(
    "missing_percent",
    ascending=False
)

display(missing_summary)

In [ ]:
display(clean_data.select_dtypes(include="number").describe().T)

In [ ]:
# checking outliers in numerical column

# Columns to check for outliers
numerical_cols = [
    "gps_height",
    "population",
    "pump_age"
]

outlier_results = []

for col in numerical_cols:

    values = clean_data[col].dropna()

    # Exclude zero population because it represents unknown population
    if col == "population":
        values = values[values != 0]

    # Calculate quartiles and IQR
    Q1 = values.quantile(0.25)
    Q3 = values.quantile(0.75)
    IQR = Q3 - Q1

    # Calculate outlier boundaries
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Identify outliers
    outliers = values[
        (values < lower_bound) | (values > upper_bound)
    ]

    outlier_results.append({
        "column": col,
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "outlier_count": len(outliers),
        "outlier_percentage": (len(outliers) / len(values)) * 100
    })

# Display outlier summary
outlier_summary = pd.DataFrame(outlier_results)

display(outlier_summary.round(2))

In [ ]:
#Visualise the outliers using boxplots

import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col in zip(axes, numerical_cols):

    sns.boxplot(
        y=clean_data[col],
        ax=ax
    )

    ax.set_title(f"Boxplot of {col}")
    ax.set_ylabel(col)

plt.tight_layout()
plt.show()

In [ ]:
# Based on decision D11, you should preserve the original longitude and latitude values in clean_data, 
# but create a separate geographical validity flag to identify coordinates that should not be used for maps 
# and distance calculations.


#Applying the agreed geographical screening range
#D11 decision specifies:
# Longitude: 29 to 41
# Latitude: -12.5 to -0.9

# Create a geographical validity flag
clean_data["geo_valid"] = (
    clean_data["longitude"].between(29, 41)
    & clean_data["latitude"].between(-12.5, -0.9)
)

# Summarise geographical screening results
print("Geographically suitable records:",
      clean_data["geo_valid"].sum())

print("Geographically unsuitable records:",
      (~clean_data["geo_valid"]).sum())

print("Missing longitude:",
      clean_data["longitude"].isna().sum())

print("Missing latitude:",
      clean_data["latitude"].isna().sum())

The codes above help to identify geographically unsuitable records without deleting or changing the original coordinates.

It checks whether each record's longitude and latitude fall within the agreed geographical ranges.
- True: Both coordinates fall within the specified ranges.
- False: At least one coordinate falls outside the ranges or is missing.
It also gives you counts of suitable and unsuitable records.

The geo_valid column identifies records that pass both coordinate checks.

In [ ]:
#inspecting unsuitable coordinates

# Select records outside the geographical screening range
unsuitable_locations = clean_data[
    ~clean_data["geo_valid"]
]

display(
    unsuitable_locations[
        ["id", "longitude", "latitude", "region", "district_code"]
    ].head(20)
)

The above helps to extract records that failed the geographical screening.

You can inspect their IDs, longitude, latitude, region and district to understand why they failed.

This helps identify potentially invalid or unusual geographical records.

In [ ]:
#Use the geographical flag for mapping
# When preparing data for maps or distance calculations, use:
geo_data = clean_data[
    clean_data["geo_valid"]
].copy()

The above creates a separate DataFrame containing only records that passed the geographical screening.

You can use this DataFrame for:
- Mapping water points.
- Calculating geographical distances.
- Analysing geographical distributions.

Original clean_data remains unchanged apart from the additional geo_valid flag.

In [ ]:
categorical_columns = clean_data.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

for col in categorical_columns:
    print(f"\n{col}")
    print(clean_data[col].value_counts(dropna=False))

In [ ]:
related_pairs = [
    ("quantity", "quantity_group"),
    ("extraction_type", "extraction_type_group"),
    ("extraction_type_group", "extraction_type_class"),
    ("management", "management_group"),
    ("payment", "payment_type"),
    ("water_quality", "quality_group"),
    ("source", "source_type"),
    ("source_type", "source_class"),
    ("waterpoint_type", "waterpoint_type_group")
]

for col1, col2 in related_pairs:
    if col1 in clean_data.columns and col2 in clean_data.columns:
        equal = clean_data[col1].eq(clean_data[col2]) | (
            clean_data[col1].isna() & clean_data[col2].isna()
        )

        print(
            f"{col1} vs {col2}: "
            f"{equal.mean() * 100:.2f}% identical"
        )

In [ ]:
#analyse how pump functionality differs across categories of categorical variables,

#For example, to examine pump functionality by water_quality:

# Cross-tabulation with counts
quality_counts = pd.crosstab(
    clean_data["water_quality"],
    clean_data["status_group"],
    dropna=False
)

# Calculate percentages within each water-quality category
quality_percentages = pd.crosstab(
    clean_data["water_quality"],
    clean_data["status_group"],
    normalize="index",
    dropna=False
) * 100

# Display sample sizes
print("Number of observations in each category:")
display(quality_counts.sum(axis=1))

print("Pump state percentages within each category:")
display(quality_percentages.round(2))

In [ ]:
#visualising

# Calculate pump-state percentages within each water-quality category
quality_percentages = pd.crosstab(
    clean_data["water_quality"].fillna("Missing"),
    clean_data["status_group"],
    normalize="index"
) * 100

# Plot
ax = quality_percentages.plot(
    kind="bar",
    stacked=True,
    figsize=(11, 6),
    width=0.75
)

ax.set_title("Pump Functionality by Water Quality")
ax.set_xlabel("Water Quality")
ax.set_ylabel("Percentage of Pumps (%)")
ax.set_ylim(0, 100)
ax.legend(title="Pump Status", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

We can do the above for other feature of choice 

In [ ]:
# Get all unique funder names and their frequencies
funder_summary = (
    clean_data["funder"]
    .value_counts(dropna=False)
    .rename_axis("funder")
    .reset_index(name="frequency")
)

print("Number of unique funder values:", clean_data["funder"].nunique(dropna=False))
print("Total funder categories, including missing:", len(funder_summary))

# Display all values without truncation
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(funder_summary)

So many errors in the name of funder e.g Ogovermnent, overnment and many more ,
how do we clean this ?

In [ ]:
installer_summary = (
    clean_data["installer"]
    .value_counts(dropna=False)
    .rename_axis("installer")
    .reset_index(name="frequency")
)

print("Number of unique installer values:", clean_data["installer"].nunique(dropna=False))
print("Total installer categories, including missing:", len(installer_summary))

# Display all values without truncation
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(installer_summary)

Recommendation
- Feature selection: Exclude funder and installer from the initial model.

In [ ]:
#cardinality
categorical_cols = train_data.select_dtypes(
    include=["object", "category", "string"]
).columns

cardinality_summary = pd.DataFrame({
    "unique_values": train_data[categorical_cols].nunique(),
    "missing_values": train_data[categorical_cols].isna().sum(),
    "missing_percent": (
        train_data[categorical_cols].isna().mean() * 100
    ).round(2)
})

cardinality_summary["unique_to_nonmissing_ratio"] = (
    cardinality_summary["unique_values"]
    / train_data[categorical_cols].count().replace(0, np.nan)
).round(3)

cardinality_summary = cardinality_summary.sort_values(
    "unique_values",
    ascending=False
)

display(cardinality_summary)

Recommendation
1. For categorical variables, do not replace missing values with the most frequent category automatically.

## Reusable model preparation and hand-off

The reusable implementation is in `src/team_wtm/model_data_preparation.py`.
It delegates cleaning and feature engineering to the shared `cleaning.py` and
`features.py` modules. Feature options and missing-feature checks therefore
have one source of truth. The notebook remains the place for audit findings and
exploratory analysis; it does not define a second split function or model.

In [ ]:
from pathlib import Path

from team_wtm.evaluation import development_folds
from team_wtm.model_data_preparation import build_preprocessor, prepare_model_inputs

X, y, record_ids = prepare_model_inputs(train_data, option="full")
assert "id" not in X.columns
assert "status_group" not in X.columns
assert X.index.equals(y.index) and X.index.equals(record_ids.index)

project_dir = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "data").is_dir() and (path / "src" / "team_wtm").is_dir()),
    None,
)
if project_dir is None:
    raise FileNotFoundError("Could not locate the team-WTM project directory.")

split_path = project_dir / "data" / "processed" / "sprint2_splits.csv"
if not split_path.exists():
    raise FileNotFoundError(
        f"Shared split plan not found at {split_path}. "
        "Use the PM-1 split plan; do not generate a notebook-specific split."
    )

split_plan = pd.read_csv(split_path)
folds = development_folds(train_data, split_plan)
fold_number, fit_rows, validation_rows = folds[0]
X_fit, X_validation = X.iloc[fit_rows], X.iloc[validation_rows]
y_fit, y_validation = y.iloc[fit_rows], y.iloc[validation_rows]

preprocessor = build_preprocessor(X_fit)
preprocessor.fit(X_fit)
print("Shared fold:", fold_number)
print("Fit rows:", len(fit_rows), "| Validation rows:", len(validation_rows))
print("Transformed feature count:", len(preprocessor.get_feature_names_out()))

In [ ]:
# Apply the same feature option and fitted preprocessing to supplied test data.
# The CSV remains local and is not added to Git.
test_path = project_dir / "data" / "raw" / "Test_set_values.csv"
if test_path.exists():
    test_values = pd.read_csv(test_path)
    X_test, _, test_ids = prepare_model_inputs(
        test_values, option="full", require_target=False
    )
    assert X_test.columns.tolist() == X.columns.tolist()
    X_test_transformed = preprocessor.transform(X_test)
    assert X_test_transformed.shape[1] == len(preprocessor.get_feature_names_out())
    assert len(test_ids) == len(X_test)
    print("Supplied test rows:", len(X_test))
else:
    print("Test_set_values.csv is not present locally; test preparation example was not run.")

## PM-4 hand-off

Import the shared preprocessor with:

```python
from team_wtm.model_data_preparation import build_preprocessor
```

Fit it inside the model pipeline using only each training fold. Use the shared
`evaluation.py` split plan and `features.py` feature options. Model construction,
cross-validation scoring and final holdout evaluation are outside PM-2 scope.